# Notebook 06: Inner Join Hub + FANTOM5

Merge the Hubstenberger RNA-seq dataset (with TPM, ~14,690 genes) with the
FANTOM5 CAGE per-gene data on Ensembl ID.

**Expected result:** 12,544 genes present in both datasets.

**Note:** NB04 and NB05 have no mutual dependency — they could run in parallel,
converging at this join step.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
REFERENCE_DIR = Path("../reference")

## 1. Load Inputs

In [2]:
# Load Hubstenberger with TPM (from NB04)
hub = pd.read_csv(PROCESSED_DIR / "04_hub_with_tpm.csv")
print(f"Hubstenberger (with TPM): {len(hub):,} genes")

# Load FANTOM5 CAGE per gene (from NB05)
cage = pd.read_csv(PROCESSED_DIR / "05_fantom5_cage_per_gene.csv")
print(f"FANTOM5 CAGE: {len(cage):,} genes")

# Check overlap before join
hub_ids = set(hub["ensembl_id"])
cage_ids = set(cage["ensembl_id"])
overlap = hub_ids & cage_ids
print(f"\nOverlap: {len(overlap):,} genes")
print(f"Hub-only: {len(hub_ids - cage_ids):,}")
print(f"CAGE-only: {len(cage_ids - hub_ids):,}")

Hubstenberger (with TPM): 15,281 genes
FANTOM5 CAGE: 15,591 genes

Overlap: 12,811 genes
Hub-only: 2,470
CAGE-only: 2,780


## 2. Inner Join

In [3]:
# Inner join on ensembl_id
merged = hub.merge(cage, on="ensembl_id", how="inner")

# Add overlap_status column (all "overlap" since it's an inner join)
merged["overlap_status"] = "overlap"

print(f"Merged dataset: {len(merged):,} genes")
print(f"Columns: {len(merged.columns)}")

Merged dataset: 12,811 genes
Columns: 35


## 3. Validation Against Jason's Reference

In [4]:
ref = pd.read_csv(REFERENCE_DIR / "12544_Hub_CAGE_MERGE_with_CapIndex.csv")

print(f"Our merged gene count: {len(merged):,}")
print(f"Jason's reference gene count: {len(ref):,}")
print(f"Match: {'YES' if len(merged) == len(ref) else 'NO'}")

# Check gene set overlap
our_ids = set(merged["ensembl_id"])
ref_ids = set(ref["ensembl_id"])
in_both = our_ids & ref_ids
in_ours_only = our_ids - ref_ids
in_ref_only = ref_ids - our_ids

print(f"\nGene set comparison:")
print(f"  In both: {len(in_both):,}")
print(f"  In ours only: {len(in_ours_only)}")
print(f"  In reference only: {len(in_ref_only)}")

if in_ours_only:
    print(f"\nExtra genes in our set:")
    for eid in sorted(in_ours_only)[:10]:
        print(f"  {eid}")

if in_ref_only:
    print(f"\nMissing genes (in reference but not ours):")
    for eid in sorted(in_ref_only)[:10]:
        gene = ref[ref["ensembl_id"] == eid]["Associated Gene Name"].values[0]
        print(f"  {eid} ({gene})")

Our merged gene count: 12,811
Jason's reference gene count: 12,544
Match: NO

Gene set comparison:
  In both: 12,290
  In ours only: 521
  In reference only: 254

Extra genes in our set:
  ENSG00000005249
  ENSG00000005700
  ENSG00000006530
  ENSG00000008018
  ENSG00000008128
  ENSG00000012963
  ENSG00000014138
  ENSG00000014164
  ENSG00000015479
  ENSG00000015568

Missing genes (in reference but not ours):
  ENSG00000000460 (C1orf112)
  ENSG00000005206 (SPPL2B)
  ENSG00000005238 (FAM214B)
  ENSG00000007202 (KIAA0100)
  ENSG00000012171 (SEMA3B)
  ENSG00000029153 (ARNTL2)
  ENSG00000035141 (FAM136A)
  ENSG00000037637 (FBXO42)
  ENSG00000047346 (FAM214A)
  ENSG00000047621 (C12orf4)


In [5]:
# Compare key numeric columns for the overlapping genes
common = merged[merged["ensembl_id"].isin(in_both)].set_index("ensembl_id")
ref_common = ref[ref["ensembl_id"].isin(in_both)].set_index("ensembl_id")

# Align on index
common = common.loc[ref_common.index]

check_cols = ["PB_TPM", "Cytosol_TPM", "NEW_Log2FC_TPM", "FANTOM_Total_CAGE_TPM", "gene_length_bp"]
check_cols = [c for c in check_cols if c in common.columns and c in ref_common.columns]

print(f"{'Column':<25} {'Pearson r':>12} {'Max diff':>12} {'Close':>8}")
print("-" * 60)
for col in check_cols:
    ours = common[col].values.astype(float)
    theirs = ref_common[col].values.astype(float)
    mask = np.isfinite(ours) & np.isfinite(theirs)
    r, _ = stats.pearsonr(ours[mask], theirs[mask])
    max_d = np.abs(ours[mask] - theirs[mask]).max()
    close = np.allclose(ours[mask], theirs[mask], rtol=1e-5, atol=1e-8)
    print(f"{col:<25} {r:>12.10f} {max_d:>12.6f} {'PASS' if close else 'FAIL':>8}")

Column                       Pearson r     Max diff    Close
------------------------------------------------------------
PB_TPM                    0.9999999991    86.884441     FAIL
Cytosol_TPM               0.9999999985    91.174887     FAIL
NEW_Log2FC_TPM            0.9999996685     0.026249     FAIL
FANTOM_Total_CAGE_TPM     0.9966245054   735.564030     FAIL
gene_length_bp            1.0000000000     0.000000     PASS


## 4. Save Output

In [6]:
output_path = PROCESSED_DIR / "06_hub_cage_merged.csv"
merged.to_csv(output_path, index=False)
print(f"Saved {len(merged):,} merged genes to {output_path}")

Saved 12,811 merged genes to ../data/processed/06_hub_cage_merged.csv
